In [1]:
import os
import json
import shutil
import pandas as pd
from sklearn.model_selection import StratifiedKFold

In [2]:
dataHum = pd.read_csv('../dataset/MultiomicsFinal.csv', index_col='ID')

In [3]:
X = dataHum.drop(columns=['Outcome at last FU'])
y = dataHum['Outcome at last FU']

duplicates = [index.split('_')[0] for index in dataHum.index if '_dp' in index]

In [22]:
def group_files_by_id(directory):
    files = os.listdir(directory)
    id_to_group = {}
    group_counter = 0

    for filename in files:
        if not filename.endswith(('.tiff', '.ndpi', '.svs')):
            continue

        splitted = filename.split('_')
        id, id_derivato = splitted[0], splitted[2]

        if 'YY-ART' in filename and id_derivato in list(dataHum.index):
            if id not in list(id_to_group.keys()) and id_derivato not in list(id_to_group.keys()):
                id_to_group[id] = group_counter
                id_to_group[id_derivato] = group_counter
                group_counter += 1
            elif id in list(id_to_group.keys()) and id_derivato not in list(id_to_group.keys()):
                id_to_group[id_derivato] = id_to_group[id]
            elif id not in list(id_to_group.keys()) and id_derivato in list(id_to_group.keys()):
                id_to_group[id] = id_to_group[id_derivato]
            else:
                pass
        else:
            if id not in list(id_to_group.keys()):
                id_to_group[id] = group_counter
                group_counter += 1
            else:
                pass

    return id_to_group

In [23]:
directory = '../dataset'
result, ids = group_files_by_id(directory)
print(result)

{'1443902': 0, '102669': 1, '146480': 2, '146504': 3, '1469819': 4, '1505177': 4, '103440': 5, '549584': 5, '1484': 6, '1487012': 7, '111878': 8, '1487180': 9, '1128767': 10, '1490244': 11, '5016288': 11, '1144687': 12, '149069': 13, '114850': 14, '115708': 15, '1494790': 16, '1739178': 16, '1161187': 17, '11717': 18, '1501188': 19, '1223647': 20, '1514045': 21, '13106895': 22, '220707': 22, '1517225': 23, '5223623': 23, '1359551': 24, '1525551': 25, '137140': 26, '1378228': 27, '1392543': 28, '152577': 29, '1418061': 30, '5182230': 30, '1549948': 31, '5116490': 31, '1557176': 32, '1418151': 13, '1570455': 33, '1815613': 33, '142782': 34, '1587240': 35, '1428187': 36, '1587990': 37, '1443593': 38, '1502873': 39, '1504203': 40, '15978': 41, '232638': 42, '68594': 42, '1604817': 43, '1608136': 44, '1613306': 45, '1832337': 45, '161664': 46, '1622752': 47, '16239': 34, '1624409': 48, '1627687': 1, '1631890': 49, '1632645': 50, '1654323': 51, '1794085': 51, '1656603': 52, '1660282': 53, '1

In [4]:
id_er = {file.split('_')[0]: file.split('_')[2] for file in os.listdir('../features') if 'YY-ART' in file}

In [17]:
'1484' in id_er.values()

True

In [5]:
with open('id_to_er.json', 'w') as fp:
    json.dump(id_er, fp)

In [6]:
duplicates = [index.split('_')[0] for index in dataHum.index if '_dp' in index]

['281766', '1820744', '5070869', '1852575', '5095313', '5230339', '2022333']

In [7]:
train_prepared = []

for id, heir in id_er.items():
    if id in list(dataHum.index) and id not in train_prepared:
        train_prepared.append(id)

    if heir in list(dataHum.index) and heir not in train_prepared:
        train_prepared.append(heir)

for id in duplicates:
    if id not in train_prepared:
        train_prepared.append(id)

    if id + '_dp' not in train_prepared:
        train_prepared.append(id + '_dp')

len(train_prepared)

164

In [8]:
X = dataHum.drop(columns=['Outcome at last FU'])
y = dataHum['Outcome at last FU']

In [9]:
y.drop(train_prepared, axis=0, inplace=True)

In [10]:
X.drop(train_prepared, axis=0, inplace=True)

In [11]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [12]:
len(train_prepared)

164

In [13]:
if os.path.exists('splits_trial'):
    shutil.rmtree('splits_trial')

os.makedirs('splits_trial')

for group, (train_index, test_index) in enumerate(skf.split(X, y)):
    train_patients = train_prepared.copy()
    train_patients.extend(X.iloc[train_index].index)
    test_patients = list(X.iloc[test_index].index)

    max_length = max(len(train_patients), len(test_patients))

    train_patients += [''] * (max_length - len(train_patients))
    test_patients += [''] * (max_length - len(test_patients))

    df = pd.DataFrame({'train': train_patients, 'val': test_patients})
    df.to_csv(f'splits_trial/{group}.csv', index=False)

    if os.path.exists(f'splits_trial/{group}.csv'):
        print(f'File number {group + 1} created!')

File number 1 created!
File number 2 created!
File number 3 created!
File number 4 created!
File number 5 created!


In [14]:
for split in os.listdir('splits_trial'):
    df = pd.read_csv(f'splits_trial/{split}')
    print(f'Split {split} has {len(df["train"]) + len(df["val"].dropna())} patients')
    print(f'Train: {len(df["train"])} --> {round(len(df["train"]) / (len(df["train"]) + len(df["val"].dropna())), 2)}')
    print(f'Val: {len(df["val"].dropna())} --> {round(len(df["val"].dropna()) / (len(df["train"]) + len(df["val"].dropna())), 2)}')
    print('')

Split 0.csv has 389 patients
Train: 344 --> 0.88
Val: 45 --> 0.12

Split 1.csv has 389 patients
Train: 344 --> 0.88
Val: 45 --> 0.12

Split 2.csv has 389 patients
Train: 344 --> 0.88
Val: 45 --> 0.12

Split 3.csv has 389 patients
Train: 344 --> 0.88
Val: 45 --> 0.12

Split 4.csv has 389 patients
Train: 344 --> 0.88
Val: 45 --> 0.12



In [15]:
shutil.rmtree('splits_trial')